In [1]:
import joblib
import numpy as np
import os
from scipy import sparse
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier

print("Loading saved engineered data...")

# === LOAD SAVED MATRICES ===
tfidf = joblib.load("../feature_engineering/tfidf_vectorizer.pkl")

X_train = joblib.load("../feature_engineering/X_train.pkl")
X_test  = joblib.load("../feature_engineering/X_test.pkl")

X_train_tfidf = joblib.load("../feature_engineering/X_train_tfidf.pkl")
X_test_tfidf  = joblib.load("../feature_engineering/X_test_tfidf.pkl")

y_train = joblib.load("../feature_engineering/y_train.pkl")
y_test  = joblib.load("../feature_engineering/y_test.pkl")

print("Data loaded successfully.\n")

Loading saved engineered data...


/Users/krishapatel/Downloads/CS439/Fraudulent-Job-Prediction/.venv/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Data loaded successfully.



/Users/krishapatel/Downloads/CS439/Fraudulent-Job-Prediction/.venv/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.3.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# === Convert booleans to numeric for sparse matrix ===
def to_numeric_sparse(df):
    df = df.copy()
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(np.int8)
    return sparse.csr_matrix(df.values), df.columns.to_numpy()

X_train_struct_sparse, struct_columns = to_numeric_sparse(X_train)
X_test_struct_sparse, _ = to_numeric_sparse(X_test)

# === Combine matrices ===
X_train_combined = sparse.hstack([X_train_tfidf, X_train_struct_sparse])
X_test_combined  = sparse.hstack([X_test_tfidf,  X_test_struct_sparse])

print("Final train shape:", X_train_combined.shape)
print("Final test shape :", X_test_combined.shape, "\n")


# =====================================================================================
#                           XGBOOST MODEL DEFINITION
# =====================================================================================

xgb = XGBClassifier(
    n_estimators=350,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",          # FAST + supports sparse matrices well
    n_jobs=-1,
    random_state=42
)


# =====================================================================================
#                           CROSS VALIDATION
# =====================================================================================

print("Running 5-fold stratified cross-validation (F1 and Accuracy)...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1 = cross_val_score(xgb, X_train_combined, y_train, cv=cv, scoring="f1")
cv_acc = cross_val_score(xgb, X_train_combined, y_train, cv=cv, scoring="accuracy")

print("\nF1 scores:", cv_f1)
print(f"Mean F1: {cv_f1.mean():.4f} | Std: {cv_f1.std():.4f}\n")

print("Accuracy scores:", cv_acc)
print(f"Mean Accuracy: {cv_acc.mean():.4f} | Std: {cv_acc.std():.4f}\n")


# =====================================================================================
#                           TRAIN FINAL MODEL
# =====================================================================================

print("Training final XGBoost model...")

xgb.fit(
    X_train_combined, y_train,
    eval_set=[(X_train_combined, y_train), (X_test_combined, y_test)],
    verbose=False
)

print("Model trained.\n")


# =====================================================================================
#                              EVALUATE MODEL
# =====================================================================================

print("Evaluating on held-out test set...\n")
y_pred = xgb.predict(X_test_combined)

print("Test Accuracy:", accuracy_score(y_test, y_pred), "\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("XGBoost – Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("xgb_confusion_matrix.png")
plt.close()
print("Confusion matrix saved as xgb_confusion_matrix.png\n")


# ================  FEATURE IMPORTANCES ===================================

print("Extracting feature importances...")

importances = xgb.feature_importances_
n_tfidf = X_train_tfidf.shape[1]

text_importances = importances[:n_tfidf]
struct_importances = importances[n_tfidf:]

feature_names_text = tfidf.get_feature_names_out()
top_text_idx = np.argsort(text_importances)[-25:][::-1]

print("\nTop TF-IDF Keywords Indicating Fraud:")
for idx in top_text_idx:
    print(f"- {feature_names_text[idx]} ({text_importances[idx]:.6f})")

top_struct_idx = np.argsort(struct_importances)[-15:][::-1]

print("\nTop Structured Features:")
for idx in top_struct_idx:
    print(f"- {struct_columns[idx]} ({struct_importances[idx]:.6f})")



# ==================== SAVE FINAL MODEL===========================================

joblib.dump(xgb, "xgboost_all_features.pkl")
print("\nModel saved as xgboost_all_features.pkl")


Final train shape: (17889, 519)
Final test shape : (7667, 519) 

Running 5-fold stratified cross-validation (F1 and Accuracy)...

F1 scores: [0.98339546 0.98371777 0.98097826 0.9843962  0.98199117]
Mean F1: 0.9829 | Std: 0.0012

Accuracy scores: [0.9863052  0.98658468 0.9843488  0.98714366 0.98518311]
Mean Accuracy: 0.9859 | Std: 0.0010

Training final XGBoost model...
Model trained.

Evaluating on held-out test set...

Test Accuracy: 0.9844789356984479 

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4461
           1       1.00      0.96      0.98      3206

    accuracy                           0.98      7667
   macro avg       0.99      0.98      0.98      7667
weighted avg       0.98      0.98      0.98      7667

Confusion matrix saved as xgb_confusion_matrix.png

Extracting feature importances...

Top TF-IDF Keywords Indicating Fraud:
- earn (0.321039)
- immediate (0.246961)
- week (0.012734)
- brin